<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/c6_ex8_ex9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MNIST dataset: train a random forest classifier, an extra-trees classifier and an SVM classifier. Then combine them into an ensemble.

In [8]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', as_frame = False)

X,y = mnist.data, mnist.target
y = y.astype(np.uint8)
print(X.shape, y.shape)

(70000, 784) (70000,)


Recall, each instance x of X is an image: an array containing 784 features, each of which is the color of the corresponding pixel (a number from 0 to 255).
y is the label, and is a number from 0 to 9.

In [9]:
# 50000 instances in X_train, 10000 in X_val and X_test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 10000, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 10000, random_state = 42)




from sklearn.ensemble import RandomForestClassifier, VotingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC

voting_clf = VotingClassifier(
    estimators=[('rf', RandomForestClassifier(n_estimators = 100, random_state = 42)),
                ('et', ExtraTreesClassifier(n_estimators = 100, random_state = 42)),
                ('smv', SVC(random_state = 42))],
    voting = 'hard'
)
voting_clf.fit(X_train, y_train)


for name, clf in voting_clf.named_estimators_.items():
    print(f"{name} score on the validation set: {clf.score(X_val, y_val)}")

print(f"Ensemble classifier (hard voting) score: {voting_clf.score(X_val, y_val)}" )


rf score on the validation set: 0.9692
et score on the validation set: 0.9715
smv score on the validation set: 0.9788
Ensemble classifier (hard voting) score: 0.9744


In [10]:
voting_clf = VotingClassifier(
    estimators=[('rf', RandomForestClassifier(n_estimators = 100, random_state = 42)),
                ('et', ExtraTreesClassifier(n_estimators = 100, random_state = 42)),
                ('smv', SVC(random_state = 42, probability=True))],
    voting = 'soft'
)
voting_clf.fit(X_train, y_train)


for name, clf in voting_clf.named_estimators_.items():
    print(f"{name} score on the validation set: {clf.score(X_val, y_val)}")

print(f"Ensemble classifier (soft voting) score: {voting_clf.score(X_val, y_val)}" )

rf score on the validation set: 0.9692
et score on the validation set: 0.9715
smv score on the validation set: 0.9788
Ensemble classifier (soft voting) score: 0.9791


In [11]:
for name, clf in voting_clf.named_estimators_.items():
    print(f"{name} score on the test set: {clf.score(X_test, y_test)}")

print(f"Ensemble classifier (soft voting) score on the test set: {voting_clf.score(X_test, y_test)}" )

rf score on the test set: 0.9645
et score on the test set: 0.9691
smv score on the test set: 0.976
Ensemble classifier (soft voting) score on the test set: 0.9767


In [20]:
# Stacking
# Get predictions from each classifier on the validation set
rf = voting_clf.named_estimators_['rf']
et = voting_clf.named_estimators_['et']
svm = voting_clf.named_estimators_['smv']

y_pred_rf = rf.predict(X_val)
y_pred_et = et.predict(X_val)
y_pred_svm = svm.predict(X_val)

# Create the new training set for the blender
X_train_blender = np.column_stack((y_pred_rf, y_pred_et, y_pred_svm))

# and the new test set
y_test_rf = rf.predict(X_test)
y_test_et = et.predict(X_test)
y_test_svm = svm.predict(X_test)

X_test_pred = np.column_stack((y_test_rf, y_test_et, y_test_svm))


X_train_blender

array([[5, 5, 5],
       [8, 8, 8],
       [2, 2, 2],
       ...,
       [7, 7, 7],
       [6, 6, 6],
       [7, 7, 7]])

Each training instance [5,5,5] is a vector that contains the predictions from each submodel.
For curiosity, let's see if we look at the probabilities outputted by the submodels.

In [22]:
y_pred_prob_rf = rf.predict_proba(X_val)
y_pred_prob_et = et.predict_proba(X_val)
y_pred_prob_svm = svm.predict_proba(X_val)

# Create the new training set for the blender
X_train_proba_blender = np.column_stack((y_pred_prob_rf, y_pred_prob_et, y_pred_prob_svm))

# and the new test set
y_test_prob_rf = rf.predict_proba(X_test)
y_test_prob_et = et.predict_proba(X_test)
y_test_prob_svm = svm.predict_proba(X_test)
X_test_predictions_prob = np.column_stack((y_test_prob_rf, y_test_prob_et, y_test_prob_svm))

X_train_proba_blender

array([[0.00000000e+00, 1.00000000e-02, 0.00000000e+00, ...,
        1.06071850e-05, 2.90600049e-04, 2.84244659e-04],
       [2.00000000e-02, 0.00000000e+00, 8.00000000e-02, ...,
        9.65145127e-07, 9.95387826e-01, 3.91280580e-05],
       [0.00000000e+00, 1.00000000e-02, 9.40000000e-01, ...,
        4.69417317e-06, 1.99049221e-04, 4.14277053e-06],
       ...,
       [0.00000000e+00, 1.00000000e-02, 0.00000000e+00, ...,
        9.96946971e-01, 2.37220946e-04, 1.34291941e-03],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.31164991e-06, 8.36900410e-07, 2.73830524e-06],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        9.99999388e-01, 1.89477753e-07, 2.83499948e-07]])

In [16]:
X_train_proba_blender[1]

array([2.00000000e-02, 0.00000000e+00, 8.00000000e-02, 8.00000000e-02,
       1.00000000e-02, 7.00000000e-02, 1.00000000e-02, 0.00000000e+00,
       7.00000000e-01, 3.00000000e-02, 1.00000000e-02, 0.00000000e+00,
       7.00000000e-02, 1.10000000e-01, 0.00000000e+00, 4.00000000e-02,
       2.00000000e-02, 0.00000000e+00, 7.40000000e-01, 1.00000000e-02,
       3.28414172e-05, 5.47689578e-06, 1.35155465e-03, 3.15446435e-03,
       1.72378608e-06, 2.41061569e-05, 1.91395660e-06, 9.65145127e-07,
       9.95387826e-01, 3.91280580e-05])

Okay, now we have created the test set for the blender. Let's train the blender.

In [24]:
rnd_forest_blender = RandomForestClassifier(n_estimators=200, oob_score=True,
                                            random_state=42)
rnd_forest_blender_prob = RandomForestClassifier(n_estimators=200, oob_score=True,
                                            random_state=42)

rnd_forest_blender.fit(X_train_blender, y_val)
rnd_forest_blender_prob.fit(X_train_proba_blender, y_val)

y_pred = rnd_forest_blender.predict(X_test_pred)
y_pred_prob = rnd_forest_blender_prob.predict(X_test_predictions_prob)

from sklearn.metrics import accuracy_score

print(f"The accuracy score of the blender trained on the hard predictions of the submodels is: {accuracy_score(y_test, y_pred)}")
print(f"The accuracy score of the blender trained on the probabilties given by the submodels is: {accuracy_score(y_test, y_pred_prob)}")

The accuracy score of the blender trained on the hard predictions of the submodels is: 0.9697
The accuracy score of the blender trained on the probabilties given by the submodels is: 0.9771


Now we use directly a Stacking Classifier

In [28]:
from sklearn.ensemble import StackingClassifier
X_train_full, y_train_full = X[:60_000], y[:60_000]
X_test, y_test = X[60_000:], y[60_000:]

estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(random_state=42))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=rnd_forest_blender, #recommened by Geron
    cv=5 # this is by default BTW
)
stacking_clf.fit(X_train_full, y_train_full)


StackingClassifier(cv=5,
                   estimators=[('rf', RandomForestClassifier(random_state=42)),
                               ('et', ExtraTreesClassifier(random_state=42)),
                               ('svm', SVC(random_state=42))],
                   final_estimator=RandomForestClassifier(n_estimators=200,
                                                          oob_score=True,
                                                          random_state=42))

In [29]:
stacking_clf.score(X_test, y_test)

0.9808